In [2]:
import os
from elasticsearch import Elasticsearch
from rag_helper import RAGBase, INSTRUCTIONS
from ingest import load_labor_code_data
from google import genai
from dotenv import load_dotenv


# 1. Load the API key from your .env file
load_dotenv()
# 1. Connect to the local Docker container
es_client = Elasticsearch('http://localhost:9200')
index_name = "juris-maroc"

# 2. Define the Elasticsearch mapping (schema)
index_settings = {
    "settings": {"number_of_shards": 1, "number_of_replicas": 0},
    "mappings": {
        "properties": {
            "content": {"type": "text"},
            "article_number": {"type": "keyword"}
        }
    }
}

# 3. Create the index and push your documents into it
es_client.indices.create(index=index_name, body=index_settings, ignore=[400])
documents = load_labor_code_data()

for doc in documents:
    es_client.index(index=index_name, document=doc)
print(f"Indexed {len(documents)} documents into Elasticsearch.")

# 4. Create the new class inheriting from RAGBase
class ElasticRAG(RAGBase):
    def __init__(self, es_client, index_name, llm_client, instructions):
        # Initialize the parent class without minsearch
        super().__init__(index=None, llm_client=llm_client, instructions=instructions)
        self.es_client = es_client
        self.index_name = index_name

    def search(self, query, num_results=5):
        # Elasticsearch specific query syntax
        search_query = {
            "size": num_results,
            "query": {
                "bool": {
                    "must": {
                        "multi_match": {
                            "query": query,
                            "fields": ["content^3", "article_number"], # Boosts content importance
                            "type": "best_fields"
                        }
                    }
                }
            }
        }

        response = self.es_client.search(index=self.index_name, body=search_query)

        # Extract the source documents from the ES JSON response
        results = [hit['_source'] for hit in response['hits']['hits']]
        return results

# 5. Test the new enterprise pipeline
client = genai.Client()
es_assistant = ElasticRAG(es_client=es_client, index_name=index_name, llm_client=client, instructions=INSTRUCTIONS)

answer = es_assistant.rag("Quels sont les droits d'une femme enceinte ?")
print(answer)

/tmp/ipykernel_8362/518760338.py:27: DeprecationWarning: Passing transport options in the API method is deprecated. Use 'Elasticsearch.options()' instead.
  es_client.indices.create(index=index_name, body=index_settings, ignore=[400])
/tmp/ipykernel_8362/518760338.py:27: DeprecationWarning: The 'body' parameter is deprecated and will be removed in a future version. Instead use individual parameters.
  es_client.indices.create(index=index_name, body=index_settings, ignore=[400])


Indexed 587 documents into Elasticsearch.


/tmp/ipykernel_8362/518760338.py:59: DeprecationWarning: The 'body' parameter is deprecated and will be removed in a future version. Instead use individual parameters.
  response = self.es_client.search(index=self.index_name, body=search_query)
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Selon le contexte fourni, les droits et dispositions concernant la femme enceinte (et la femme en général) sont les suivants :

1. **Suivi médical (Article 327) :** Dans les entreprises soumises à l'obligation de disposer d'un service médical du travail, la femme enceinte doit faire l'objet d'un examen médical par le médecin du travail, suivant une fréquence dont ce médecin reste juge.

2. **Droits généraux et non-discrimination (Article 9) :** 
   * L'interdiction de toute discrimination fondée sur le sexe ou la situation conjugale (notamment en matière d'embauchage, de salaire, de conditions de travail ou de licenciement).
   * Le droit de conclure un contrat de travail.
   * Le droit, qu'elle soit mariée ou non, d'adhérer à un syndicat professionnel et de participer à son administration et à sa gestion.


In [3]:
from google.genai import types

search_tool = types.Tool(
    function_declarations=[
        types.FunctionDeclaration(
            name="search",
            description="Recherche dans la base de données du Code du Travail marocain pour trouver les articles pertinents.",
            parameters=types.Schema(
                type=types.Type.OBJECT,
                properties={
                    "query": types.Schema(
                        type=types.Type.STRING,
                        description="Le texte de la requête de recherche juridique."
                    )
                },
                required=["query"]
            )
        )
    ]
)

In [4]:
from google.genai import types
from tenacity import retry, stop_after_attempt, wait_exponential

class AgenticElasticRAG:
    def __init__(self, es_client, index_name, llm_client, instructions):
        self.es_client = es_client
        self.index_name = index_name
        self.llm_client = llm_client
        self.instructions = instructions

        # 1. Initialize the Chat Session with your tool
        self.chat_session = self.llm_client.chats.create(
            model= "gemini-3.6-flash",
            config=types.GenerateContentConfig(
                system_instruction=self.instructions,
                tools=[self.search_juris_maroc], # We pass the method directly
                temperature=0.1 # Kept low for factual legal accuracy
            )
        )

    # 2. Define the tool with explicit Type Hints and a Docstring.
    # Gemini reads this metadata to understand HOW and WHEN to use the tool.
    def search_juris_maroc(self, query: str) -> str:
        """Recherche dans la base de données du Code du Travail marocain pour trouver les articles pertinents.

        Args:
            query: Le texte de la requête de recherche juridique.
        """
        print(f"\n[Outil Agent] L'agent a décidé de chercher : '{query}'")

        search_query = {
            "size": 5,
            "query": {
                "bool": {
                    "must": {
                        "multi_match": {
                            "query": query,
                            "fields": ["content^3", "article_number"],
                            "type": "best_fields"
                        }
                    }
                }
            }
        }

        response = self.es_client.search(index=self.index_name, body=search_query)
        results = [hit['_source']['content'] for hit in response['hits']['hits']]

        # Return a single concatenated string for Gemini to read
        return "\n\n---\n\n".join(results)

    # 3. The execution method with your network retry shield
    @retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=1, min=2, max=10))
    def ask(self, user_message: str):
        # AFC handles the loop: it receives the message, triggers the tool if needed, and returns the final text.
        response = self.chat_session.send_message(user_message)
        return response.text

# --- Test the new Agent ---
# Assuming es_client and client are already initialized in your notebook from the previous steps
agent = AgenticElasticRAG(
    es_client=es_client,
    index_name=index_name,
    llm_client=client,
    instructions=INSTRUCTIONS
)

# Ask a conversational question
print("Utilisateur: Bonjour !")
print(f"Agent: {agent.ask('Bonjour !')}")

# Ask a complex legal question to trigger the tool
print("\nUtilisateur: Peux-tu m'expliquer les conditions de la période d'essai ?")
final_answer = agent.ask("Peux-tu m'expliquer les conditions de la période d'essai ?")
print(f"\nAgent:\n{final_answer}")

Utilisateur: Bonjour !
Agent: Bonjour ! Comment puis-je vous aider aujourd'hui concernant le droit du travail marocain ?

Utilisateur: Peux-tu m'expliquer les conditions de la période d'essai ?

[Outil Agent] L'agent a décidé de chercher : 'période d'essai durée renouvellement rupture'


/tmp/ipykernel_8362/826938560.py:46: DeprecationWarning: The 'body' parameter is deprecated and will be removed in a future version. Instead use individual parameters.
  response = self.es_client.search(index=self.index_name, body=search_query)



Agent:
Selon le Code du Travail marocain (Loi 65-99), voici les conditions et règles régissant la période d'essai :

### 1. Définition et principe de rupture (Article 13)
* **Principe :** La période d'essai est la période pendant laquelle chacune des parties peut rompre volontairement le contrat de travail sans préavis ni indemnité.
* **Délai de préavis en cas de rupture :** Après au moins une semaine de travail, la rupture non motivée par une faute grave du salarié nécessite le respect d'un préavis :
  * **2 jours** avant la rupture si le salarié est payé à la journée, à la semaine ou à la quinzaine ;
  * **8 jours** avant la rupture s'il est payé au mois.

---

### 2. Durée de la période d'essai (Article 14)

#### **Pour les Contrats à Durée Indéterminée (CDI) :**
* **3 mois** pour les cadres et assimilés ;
* **1 mois et demi** pour les employés ;
* **15 jours** pour les ouvriers.

*La période d'essai pour un CDI peut être renouvelée **une seule fois**.*

#### **Pour les Contrats à 